<div align="center">

# 🧠 **CBS5502 — Computational Linguistics and NLP Technologies**

### 🐍 **2nd Python Tutorial**
### 📅 *February 4, 2026*

---

## 🇨🇳 **PoS Tagging and Disambiguation**
---
### 👨‍🏫 **Instructor**
**Dr. WAN Mingyu**

### 👨‍🏫 **Teaching Assistant**
**Mr. BAO Xiaoyi**

</div>

---

## 🌟 Welcome!

Welcome to the tutorial series of **CBS5502**!  
In this tutorial, we will explore how **Part-of-Speech (PoS) tagging** works and how ambiguity can be resolved using **three different approaches**, all demonstrated with the classic ambiguous sentence:

> **“We can can the can.”** 🌟

---

## 🎯 Learning Objectives

By the end of this tutorial, you will be able to:

- Understand what **PoS tagging** is and why it is important in NLP  
- Identify **lexical and structural ambiguity** in natural language  
- Apply **three approaches to PoS tagging**:
  - Rule-based tagging
  - Statistical / probabilistic tagging
  - Dictionary‑ or library‑based tagging using Python  
- Analyze and interpret tagging results for ambiguous sentences  

---

🚀 Let’s Get Started!

In [4]:
# Import required libraries
import nltk
nltk.download('punkt_tab')
from nltk.probability import FreqDist
from nltk.util import ngrams
from nltk.tag import hmm
from collections import defaultdict
from nltk.tag import brill, brill_trainer
from nltk.tag import UnigramTagger, BigramTagger, DefaultTagger
from nltk.corpus import treebank

# Ensure you have the required NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Input sentence
sentence = "We can can the can."

# Tokenizing the sentence into words
tokens = nltk.word_tokenize(sentence)
print("Tokenized Sentence:", tokens)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Tokenized Sentence: ['We', 'can', 'can', 'the', 'can', '.']


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


### 1️⃣ Rule‑Based Approach

#### 🔍 Overview
The **rule‑based approach** assigns Part‑of‑Speech (PoS) tags using **handcrafted linguistic rules**, typically based on word forms, surrounding context, or fixed patterns.  
This method does **not rely on training data**, making it easy to understand and implement.

#### 🧠 How It Works
- Each rule matches a word (or pattern) in the sentence
- The **first matching rule** determines the PoS tag
- Rules are applied **sequentially**, from top to bottom

#### 🧩 Example Rules
For our ambiguous sentence, we define a few **simple and intuitive rules**:
- Tag **“We”** as a personal pronoun
- Tag **“the”** as a determiner
- Assign **“can”** a default modal‑verb tag
- Use a fallback rule for unknown cases

These rules illustrate both the **strength** (clarity) and **limitation** (lack of context awareness) of the rule‑based approach.

In [1]:
# --------------------------------------------------
# STEP 1: Define default (most likely) POS tags
# --------------------------------------------------
# This dictionary provides a fallback tag for each word.
# If no contextual rule applies, we use these tags.
most_likely_tags = {
    "We": "PRP",   # Personal pronoun
    "can": "MD",   # Modal verb (default assumption)
    "the": "DT"    # Determiner
}

In [2]:
# --------------------------------------------------
# STEP 2: Define the rule-based POS tagging function
# --------------------------------------------------
def rule_based_pos_tagger(tokens):
    """
    Assign POS tags to a list of tokens using
    handcrafted contextual rules.

    Parameters:
        tokens (list): A list of word tokens

    Returns:
        list: A list of (word, POS tag) tuples
    """

    tagged_sentence = []  # Store the final tagged output

    # Iterate through each word with its position
    for i, word in enumerate(tokens):

        # --------------------------------------------------
        # STEP 3: Apply context-sensitive rules
        # --------------------------------------------------

        # Rule 1:
        # If "can" appears immediately after "We",
        # it functions as a modal verb (e.g., "We can ...")
        if word == "can" and i > 0 and tokens[i - 1] == "We":
            tag = "MD"

        # Rule 2:
        # If "can" follows "the", it is treated as a noun
        # (e.g., "the can")
        elif word == "can" and i > 0 and tokens[i - 1] == "the":
            tag = "NN"

        # Rule 3:
        # If "can" follows another "can",
        # it is treated as a main verb
        # (e.g., "can can the...")
        elif word == "can" and i > 0 and tokens[i - 1] == "can":
            tag = "VB"

        # --------------------------------------------------
        # STEP 4: Apply default rule
        # --------------------------------------------------
        # If no specific contextual rule matches,
        # fall back to the most likely tag
        else:
            tag = most_likely_tags.get(word, "NN")
            # Unknown words default to NN (noun)

        # Add the (word, tag) pair to the result
        tagged_sentence.append((word, tag))

    return tagged_sentence

## 🔎 Step‑by‑Step Rule Application

Sentence: **We can can the can .**

| Position | Token | Left Context | Applied Rule | Assigned Tag |
|---------:|-------|--------------|--------------|--------------|
| 0 | We | — | Default dictionary rule | PRP |
| 1 | can | We | Rule 1: *can* after *We* | MD |
| 2 | can | can | Rule 3: *can* after *can* | VB |
| 3 | the | can | Default dictionary rule | DT |
| 4 | can | the | Rule 2: *can* after *the* | NN |
| 5 | . | can | Default fallback | NN |

In [5]:
# --------------------------------------------------
# STEP 5: Apply the rule-based tagger
# --------------------------------------------------
rule_based_tags = rule_based_pos_tagger(tokens)

# Display the result
print("Rule-Based POS Tags:")
for word, tag in rule_based_tags:
    print(f"{word:>5}  →  {tag}")

Rule-Based POS Tags:
   We  →  PRP
  can  →  MD
  can  →  VB
  the  →  DT
  can  →  NN
    .  →  NN


## ⚠️ Error Cases & Discussion

### Example 1
Sentence: **They can fish.**

Expected:
- can → MD
- fish → VB

Rule-Based Output:
- can → MD ✅
- fish → NN ❌

📌 *Why?*  
The system lacks a rule recognizing **verb usage without “the”**.

### Example 2
Sentence: **The can can rust.**

Correct interpretation:
- can → NN
- can → VB

Rule-Based Output:
- can → NN ✅
- can → VB ✅ (by coincidence)

📌 *Discussion point:*  
Correct tagging here is **accidental**, not robust.

### 🧠 Teaching Notes

- This approach relies entirely on **manually written rules**
- Each rule encodes **explicit linguistic intuition**

#### ✅ Strengths
- Easy to understand and interpret
- Transparent decision‑making process

#### ❌ Limitations
- Difficult to scale to large vocabularies
- Brittle when encountering unseen or unexpected patterns

### 🧠 Learning Takeaways

- Rule-based tagging makes **linguistic assumptions explicit**
- Context helps, but only when **manually encoded**
- Error cases reveal why **learning from data is necessary**
- HMM and Brill taggers automate what rules attempt to approximate

## 2️⃣ Hidden Markov Model (HMM) Approach

### 🔍 Overview
The **Hidden Markov Model (HMM)** approach is a **statistical sequence‑labeling method** that assigns PoS tags by modeling language as a **probabilistic process**.  
It predicts the **most likely sequence of tags** for a sentence using probabilities learned from a **tagged corpus**.

### 🧠 Core Assumptions
HMM PoS tagging relies on two key assumptions:

- **Markov Assumption:**  
  The current tag depends only on a limited number of previous tags (typically one or two).
- **Output Independence Assumption:**  
  Each word is generated independently given its tag.

### 🔁 Decoding Strategy
To determine the optimal tag sequence, HMMs use the **Viterbi algorithm**, which efficiently finds:

> ✅ The most probable tag sequence for the entire sentence,  
> rather than tagging each word independently.



### 🧠 Step 1: What Does an HMM Model?

An HMM models language with two probability components:

1. **Transition Probability**
   - $$P(t_i \mid t_{i-1})$$  
   - How likely one tag follows another

2. **Emission Probability**
   - $$P(w_i \mid t_i)$$  
   - How likely a word is generated by a tag

The goal is to find the **most probable tag sequence** for the entire sentence.

### 🔁 Step 2: Why We Need Sequence‑Level Decisions

Ambiguous words like **“can”** cannot be tagged reliably in isolation.

✅ HMMs solve this by:
- Considering **previous tags**
- Evaluating the **entire sentence**
- Using **global optimization** via the Viterbi algorithm

### 🔧 Step 3: Environment Setup

In [6]:
import nltk
from nltk.corpus import brown
from nltk.tag import hmm

In [7]:
# Download resources (run once)
nltk.download('punkt')
nltk.download('brown')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

### 📚 Step 4: Prepare Training Data

HMMs require a **tagged corpus** to learn probabilities.
Here, we use the Brown Corpus (news category).

## 🏷️ Official Tag List for `hmm_tagger` (Brown Tagset)

### ✅ Primary Reference (Recommended for Teaching)

You can find the **complete Brown Corpus tagset**, along with detailed explanations, in the **NLTK Book, Chapter 5**:

👉 **NLTK Book — Categorizing and Tagging Words (Brown Tagset)**  
*(nltk.org)*

### 📚 What This Section Covers

This reference documents:

- ✅ All Brown tags (e.g. `PPSS`, `AT`, `NP`, `VB`, `MD`)
- ✅ Examples of words annotated with each tag
- ✅ Key differences between the **Brown tagset** and the **Penn Treebank tagset**

📌 **Teaching note:**  
The `HiddenMarkovModelTagger` in NLTK inherits its tagset directly from the corpus it is trained on. When trained with the Brown Corpus, it therefore produces **Brown-style PoS tags**.

In [8]:
# Load tagged sentences for training
train_sentences = brown.tagged_sents(categories='news')

# Inspect one example
train_sentences[0]

[('The', 'AT'),
 ('Fulton', 'NP-TL'),
 ('County', 'NN-TL'),
 ('Grand', 'JJ-TL'),
 ('Jury', 'NN-TL'),
 ('said', 'VBD'),
 ('Friday', 'NR'),
 ('an', 'AT'),
 ('investigation', 'NN'),
 ('of', 'IN'),
 ("Atlanta's", 'NP$'),
 ('recent', 'JJ'),
 ('primary', 'NN'),
 ('election', 'NN'),
 ('produced', 'VBD'),
 ('``', '``'),
 ('no', 'AT'),
 ('evidence', 'NN'),
 ("''", "''"),
 ('that', 'CS'),
 ('any', 'DTI'),
 ('irregularities', 'NNS'),
 ('took', 'VBD'),
 ('place', 'NN'),
 ('.', '.')]

### 🏗️ Step 5: Train the HMM Tagger

The training process automatically learns:
- Tag transition probabilities
- Word emission probabilities

In [9]:
# Train an HMM tagger
hmm_tagger = hmm.HiddenMarkovModelTagger.train(train_sentences)

### 🔍 Step 6: Apply the HMM to an Ambiguous Sentence

In [11]:
sentence = "We can can the can ."
tokens = nltk.word_tokenize(sentence)

hmm_tags = hmm_tagger.tag(tokens)
hmm_tags

[('We', 'PPSS'),
 ('can', 'MD'),
 ('can', 'VB'),
 ('the', 'AT'),
 ('can', 'NN'),
 ('.', '.')]

✔ The HMM correctly captures grammatical structure  
✔ Ambiguity is resolved using contextual probabilities  
✔ The same word can have different tags in the same sentence

## 3️⃣ Transformation‑Based (Brill) Tagging

### 🔍 Overview
The **Transformation‑Based Approach**, also known as **Brill Tagging**, is a **hybrid method** that combines:

- ✅ A **simple statistical baseline tagger**
- ✅ A set of **learned transformation rules**

Instead of assigning tags in one step, Brill tagging **iteratively corrects errors** made by an initial tagger using rules learned from a **tagged corpus**.


### 🧠 Core Idea

Brill tagging follows this learning cycle:

1. Start with a **baseline tagger** (e.g., Unigram Tagger)
2. Compare its output with **gold‑standard tags**
3. Learn **transformation rules** that reduce errors
4. Apply the rules sequentially to improve tagging accuracy

📌 The learned rules are **human‑readable**, making this approach both **accurate and interpretable**.

### 🔧 Step 1: Environment Setup

In [26]:
!pip install -q sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.9 MB/s eta 0:00:00


In [27]:
import nltk
from nltk.corpus import brown

nltk.download('brown')
nltk.download('punkt')

sentences = brown.tagged_sents(categories='news')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [28]:
from nltk.tag import DefaultTagger, UnigramTagger

default_tagger = DefaultTagger('NN')
baseline_tagger = UnigramTagger(sentences, backoff=default_tagger)

In [29]:
sentence = "We can can the can .".split()
baseline_tagger.tag(sentence)

[('We', 'PPSS'),
 ('can', 'MD'),
 ('can', 'MD'),
 ('the', 'AT'),
 ('can', 'MD'),
 ('.', '.')]

In [30]:
def word_features(sent, i):
    word = sent[i]
    features = {
        'word.lower()': word.lower(),
        'is_upper': word.isupper(),
        'is_title': word.istitle(),
        'is_digit': word.isdigit(),
    }
    if i > 0:
        features['prev_word'] = sent[i-1]
    else:
        features['BOS'] = True

    if i < len(sent) - 1:
        features['next_word'] = sent[i+1]
    else:
        features['EOS'] = True

    return features

In [31]:
def sent2features(sent):
    return [word_features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for _, tag in sent]

X = [sent2features([w for w, t in s]) for s in sentences]
y = [sent2labels(s) for s in sentences]

In [34]:
import sklearn_crfsuite

# Use a manageable subset for quick training
X_small = X[:500]
y_small = y[:500]

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=20,
    all_possible_transitions=False
)

crf.fit(X_small, y_small)
print("CRF model trained on", len(X_small), "sentences")

CRF(algorithm='lbfgs', all_possible_transitions=False, max_iterations=10)

In [35]:
test_sentence = "We can can the can .".split()
features = sent2features(test_sentence)

list(zip(test_sentence, crf.predict_single(features)))

[('We', 'DT'),
 ('can', 'NN'),
 ('can', 'IN'),
 ('the', 'AT'),
 ('can', 'NN'),
 ('.', '.')]

### ⚠️ Practical Note on CRF Training

CRFs are powerful but computationally expensive models.
Training on large corpora with all possible tag transitions
can take a very long time.

✅ For teaching and experimentation:
- Use a small subset of data
- Limit the number of iterations
- Disable unnecessary transitions

This preserves the learning behavior while keeping runtime manageable.

#### 🧠 Interpretation

- **Rule‑Based Tagger**
  - Correctly handles this sentence due to carefully designed rules.
  - Performance is **fragile** and depends entirely on manual rule coverage.

- **Hidden Markov Model (HMM)**
  - Resolves ambiguity using **learned transition and emission probabilities**.
  - Makes **global sequence‑level decisions**, leading to robust results.

- **Brill‑Style (Transformation‑Based) Tagger**
  - Starts from a weak baseline and **learns contextual corrections**.
  - Combines the **interpretability of rules** with **data‑driven learning**.
  - Often outperforms unigram or bigram taggers when **training data is limited**.

---

#### ✅ Key Takeaway

Although all three methods succeed on this example, they do so for different reasons:

- Rule‑based tagging relies on **explicit linguistic intuition**
- HMM tagging relies on **probabilistic sequence modeling**
- Brill‑style tagging bridges both worlds by **learning rules from data**

This comparison highlights why transformation‑based methods remain an important conceptual bridge between symbolic and statistical NLP approaches.

## 📝 Playground — Ending Exercises

The following exercises encourage you to **apply, compare, and reflect** on the three PoS tagging approaches covered in this tutorial. Focus on **ambiguity**, **context**, and **model behavior** rather than just correctness.

---

### 🧪 Exercise 1: English Ambiguity Challenge  
**Sentence:**  
> *“Time flies like an arrow.”*

This sentence is famously ambiguous and can be interpreted in multiple ways.

#### ✅ Tasks
1. **Tokenize** the sentence.
2. Apply:
   - Rule‑based tagging  
   - HMM tagging  
   - Brill‑style (transformation‑based) tagging
3. Record the PoS tags produced by each method.

#### 🧠 Guiding Questions
- Which word(s) show different PoS tags across methods?
- Does *flies* behave as a **noun** or a **verb**?
- Is *like* treated as a **verb**, **preposition**, or **conjunction**?
- Which approach best captures the intended reading?

#### 💡 Reflection
- Why is this sentence difficult to tag correctly without full syntactic analysis?
- How does sequence‑level modeling help resolve ambiguity?

---

### 🧪 Exercise 2: Chinese Structural Ambiguity  
**Sentence:**  
> **我喜欢吃苹果的人。**  
> *(Pinyin: Wǒ xǐhuān chī píngguǒ de rén.)*

This sentence is a classic example used in **Chinese NLP** to test ambiguity resolution.

#### ✅ Tasks
1. Segment the sentence into words (use a Chinese tokenizer if available).
2. Assign PoS tags to each word.
3. Identify at least **two possible interpretations** of the sentence.

#### 🧠 Key Points to Consider
- The grammatical role of **“的”**
- Whether **“吃苹果”** modifies:
  - *我* (I like to eat apples), or
  - *人* (people who eat apples)
- How relative clauses are formed in Chinese

#### 💡 Reflection
- Why is **“的”** challenging for PoS tagging and parsing?
- What additional information (syntax, semantics, or context) would help disambiguate the sentence?
- Why do purely rule‑based approaches struggle with this example?

---

### 🌟 Take‑Home Insight

These exercises illustrate that:

- **PoS tagging alone is often insufficient** for full disambiguation
- Ambiguity exists at both **lexical** and **structural** levels
- Real‑world NLP systems must integrate **context, syntax, and semantics**

✅ Congratulations on completing the tutorial!

In [ ]:
# ===== Exercise Solutions =====
# Exercise 1: English ambiguity sentence
amb_sent = "Time flies like an arrow ."
amb_tokens = nltk.word_tokenize(amb_sent)

print("Tokens:", amb_tokens)
print("\nRule-based (demo):", rule_based_pos_tagger(amb_tokens))
print("HMM:", hmm_tagger.tag(amb_tokens))
print("CRF-like baseline:", baseline_tagger.tag(amb_tokens))

print("\nQuick analysis:")
print("- 'flies' is usually tagged as VBZ in this context (verb reading).")
print("- 'like' is often tagged as IN (preposition).")
print("- Different taggers may disagree when context is short or ambiguous.")

# Exercise 2: Chinese structural ambiguity
zh_sent = "我喜欢吃苹果的人。"
try:
    import jieba.posseg as pseg
    zh_tags = [(w.word, w.flag) for w in pseg.cut(zh_sent)]
    print("\nChinese segmentation + POS:", zh_tags)
except Exception as e:
    print("\nJieba POS unavailable:", e)

print("\nTwo possible interpretations:")
print("1) 我 喜欢 [吃苹果 的 人]  -> I like people who eat apples.")
print("2) [我 喜欢 吃苹果] 的 人   -> people who like eating apples (speaker-including reading).")
print("\nWhy difficult: '的' links modifiers to nouns, and ambiguity needs syntax + semantics.")